# Model Benchmark Results — Pharma-DemandForecast

Leaderboard and analysis across all trained models.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

RESULTS_DIR = Path('../results')

## 1. Load All Results

In [ ]:
records = []
for metrics_file in sorted(RESULTS_DIR.glob('*/metrics.json')):
    with open(metrics_file) as f:
        records.append(json.load(f))

if not records:
    print('No results found. Run: make train CONFIG=<config_path> to generate results.')
else:
    leaderboard = pd.DataFrame(records)
    print(f'Loaded {len(leaderboard)} runs.')
    print(leaderboard.columns.tolist())

## 2. Leaderboard Table (ranked by RMSSE val2)

In [ ]:
if records:
    display_cols = ['run_id', 'model', 'strategy',
                    'wmape_val1', 'rmsse_val1', 'wmape_val2', 'rmsse_val2', 'runtime_seconds']
    available = [c for c in display_cols if c in leaderboard.columns]
    lb = leaderboard[available].sort_values('rmsse_val2').reset_index(drop=True)
    lb.index += 1  # 1-based ranking

    # Format
    for col in ['wmape_val1', 'rmsse_val1', 'wmape_val2', 'rmsse_val2']:
        if col in lb:
            lb[col] = lb[col].map('{:.4f}'.format)
    if 'runtime_seconds' in lb:
        lb['runtime_seconds'] = lb['runtime_seconds'].map('{:.0f}s'.format)

    pd.set_option('display.max_colwidth', 40)
    display(lb)

## 3. WMAPE vs RMSSE Scatter

In [ ]:
if records:
    family_map = {
        'naive': 'Baseline', 'croston': 'Baseline',
        'ets': 'Statistical', 'theta': 'Statistical', 'autoarima': 'Statistical',
        'lgbm': 'Gradient Boosting', 'xgb': 'Gradient Boosting',
        'nbeats': 'Deep Learning', 'nhits': 'Deep Learning',
        'tft': 'Deep Learning', 'patchtst': 'Deep Learning',
        'ensemble': 'Ensemble',
    }
    lb_num = leaderboard.copy()
    for col in ['wmape_val2', 'rmsse_val2']:
        if col in lb_num:
            lb_num[col] = pd.to_numeric(lb_num[col], errors='coerce')
    lb_num['family'] = lb_num['model'].map(family_map).fillna('Other')

    palette = {
        'Baseline': '#1f77b4', 'Statistical': '#ff7f0e',
        'Gradient Boosting': '#2ca02c', 'Deep Learning': '#d62728',
        'Ensemble': '#9467bd', 'Other': '#7f7f7f',
    }

    fig, ax = plt.subplots(figsize=(9, 6))
    for family, grp in lb_num.groupby('family'):
        ax.scatter(grp['wmape_val2'], grp['rmsse_val2'],
                   label=family, color=palette.get(family, 'gray'), s=80, alpha=0.8)
    for _, row in lb_num.iterrows():
        if pd.notna(row.get('wmape_val2')) and pd.notna(row.get('rmsse_val2')):
            label = f"{row['model']}{'_' + row['strategy'] if row.get('strategy') else ''}"
            ax.annotate(label, (row['wmape_val2'], row['rmsse_val2']),
                        textcoords='offset points', xytext=(4, 4), fontsize=7)
    ax.set_xlabel('WMAPE (val2)')
    ax.set_ylabel('RMSSE (val2)')
    ax.set_title('WMAPE vs RMSSE — Val Fold 2')
    ax.legend(title='Model Family')
    plt.tight_layout()
    plt.show()

## 4. Error Distribution by Model Family

In [ ]:
if records and 'rmsse_val2' in lb_num.columns:
    lb_num['rmsse_val2_num'] = pd.to_numeric(lb_num['rmsse_val2'], errors='coerce')
    fig, ax = plt.subplots(figsize=(10, 5))
    order = lb_num.groupby('family')['rmsse_val2_num'].median().sort_values().index
    sns.boxplot(data=lb_num, x='rmsse_val2_num', y='family', order=order,
                palette=palette, ax=ax, orient='h')
    ax.set_xlabel('RMSSE (val2)')
    ax.set_ylabel('Model Family')
    ax.set_title('RMSSE Distribution by Model Family')
    plt.tight_layout()
    plt.show()

## 5. Best Model — Actual vs Predicted (4 representative series)

In [ ]:
if records:
    lb_sorted = leaderboard.copy()
    lb_sorted['rmsse_val2_num'] = pd.to_numeric(lb_sorted.get('rmsse_val2', np.nan), errors='coerce')
    best_run_id = lb_sorted.sort_values('rmsse_val2_num').iloc[0]['run_id']
    pred_path = RESULTS_DIR / best_run_id / 'predictions_val2.parquet'
    actual_path = Path('../data/processed/long.parquet')

    if pred_path.exists() and actual_path.exists():
        preds = pd.read_parquet(pred_path)
        long_df = pd.read_parquet(actual_path)

        # Select 4 representative series: one per category + most intermittent
        cat_series = long_df[['id','cat_id']].drop_duplicates().groupby('cat_id', observed=True).first()['id']
        series_ids = cat_series.tolist()[:4]

        fig, axes = plt.subplots(2, 2, figsize=(16, 8))
        for ax, sid in zip(axes.flat, series_ids):
            actual = long_df[(long_df['id'] == sid) & (long_df['d'].between(1830, 1913))]
            pred_s = preds[preds['id'] == sid] if 'id' in preds.columns else pd.DataFrame()

            ax.plot(pd.to_datetime(actual['date']), actual['sales'], label='Actual', color='steelblue')
            if not pred_s.empty and 'yhat' in pred_s.columns:
                ax.plot(pd.to_datetime(pred_s['date']), pred_s['yhat'],
                        label='Forecast', color='tomato', linestyle='--')
            ax.set_title(sid[:40])
            ax.legend(fontsize=8)
            ax.tick_params(axis='x', rotation=30)

        plt.suptitle(f'Best model ({best_run_id}) — Val Fold 2 forecasts', fontsize=12)
        plt.tight_layout()
        plt.show()
    else:
        print(f'Prediction file not found: {pred_path}')

## 6. LightGBM Feature Importance

In [ ]:
# Load feature importance from the best LGBM run (if available)
lgbm_runs = [r for r in records if r.get('model') == 'lgbm']
if lgbm_runs:
    print('Feature importance is logged per run. Re-fit the model and call:')
    print('    model.get_feature_importance().head(20).plot.barh()')
    print('\nOr MLflow UI shows feature importance for LGBM runs automatically.')
else:
    print('No LGBM runs found yet.')

## 7. Ensemble vs Individual Models

In [ ]:
if records:
    lb_num['is_ensemble'] = lb_num['model'] == 'ensemble'
    ensemble_runs = lb_num[lb_num['is_ensemble']]
    individual_runs = lb_num[~lb_num['is_ensemble']]

    if not ensemble_runs.empty:
        best_individual = individual_runs['rmsse_val2_num'].min()
        for _, row in ensemble_runs.iterrows():
            improvement = (best_individual - row['rmsse_val2_num']) / best_individual * 100
            print(f"Ensemble ({row['strategy']}): RMSSE={row['rmsse_val2_num']:.4f}  "
                  f"({'improved' if improvement > 0 else 'worse'} by {abs(improvement):.1f}% vs best individual)")
    else:
        print('No ensemble runs found. Train with configs/ensemble_average.yaml after individual models.')